# Esse é o tratamento de todos os arquivos


## Carregando as variáveis de ambiente
*Estou carregando pelo pandas e não pelo numpy. Por que? Porque eu quero!*

In [1]:
import pandas as pd
import numpy as np
import filter as ft
import IPython.display as dsp
import scipy as sc
import seaborn as sn
import utils2 as ut
import analysis as an
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json
import warnings
from configuration_new import Configuration
from vars import PROFILE_BANDS, __version__, OPEN_LIMITATIONS, BLOCKED_ANALYSES
from typing import Dict, Any
from sklearn.metrics import (roc_auc_score)
from dataclasses import asdict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## Criando as variáveis iniciais

In [2]:
cfg = Configuration()
path_data = "data/"
meta_file = "data/meta.xlsx"
path_output =  'output'
path_output =  f"output_{datetime.now():%Y_%m_%d_%H_%M_%S}/"
path_output =  f"lixo/"
output = Path(path_output)
output_directories = {"integration":None, "quality":None, "analysis":None, "sensitivity":None}
for key in output_directories:
    output_directories[key]=output/key
    output_directories[key].mkdir(parents=True, exist_ok=True)

loaded_files = {} #recs
filtered_data = {}
failures=[]
sampling_verification = [ ]# fsv_rows
quality_control, profiles_rows = [], []
selection_criterion = "55 registros de repouso do experimento de 3 eletrodos do MODMA, lote completo, sem seleção adicional."

## Carregando arquivos e informações do dataset (metadados)

In [3]:
metadata = ut.load_modma_metadata(meta_file)
files, failures_files = ut.discover_eeg_txt_files(path_data)

## Fazendo a análise dos arquivos

In [4]:
for file in files:
    try:
        
        rec = ut.load_modma_txt(file,cfg, file.name)

        loaded_files[rec.subject_id] = rec


        fsv = ut.verify_sampling_rate(rec.data_counts, cfg)

        fsv['subject_id'] = rec.subject_id

        sampling_verification.append(fsv)

        subject_id, subject_state = file.name.split('.')[0].split('_')

        data_raw = pd.read_csv(file, sep=" ", names=cfg.electrode_labels)

        data_raw = np.transpose(data_raw.to_numpy().astype(float)) #Convertendo os dados para np.ndarray
        
        data_np, changed_values_on_wraparound = ft.fix_integer_wraparound(data_raw)

        data_np = np.ascontiguousarray(data_np)

        data_np, filter_info = ft.apply_filter(data_np, cfg) 

        filtered_data[subject_id] = data_np

        profile = ut.spectral_profile_blocked(data_np, cfg.sampling_frequency, cfg.block_time)

        profile['subject_id'] = subject_id

        profile['task'] = subject_state

        profiles_rows.append(profile)

        metrics = ut.quality_control_metrics(data_raw, data_np, filtering_info=filter_info, subject_id=subject_id, file_name=file.name, eeg_task=subject_state, n_wraparound=changed_values_on_wraparound)

        quality_control.append(metrics)
        
    except Exception as error:
        print(f"Failed to process file {file.name}: {error}")
        failures_files.append({
                "file": file.name, 
                "subject_id": file.stem.split("_")[0],
                "stage": "read_or_profile",
                 "error": "%s: %s" % (type(error).__name__, str(error)[:200])})

# Calculando a Coorte nos blocos terminais

In [5]:
# print(profiles_rows)
cohort_reference = an.cohort_terminal_profile(profiles_rows, cfg)
print(cohort_reference) # Ver essa diferença - possivelmente é o task=still

[0.52443225 0.14601433 0.19590037 0.11376119 0.01989186]


# Calculando o tempo de estabilização

In [6]:
settle_rows = []
# print(results)
for prof in profiles_rows:
    sid = str(prof["subject_id"].iloc[0])
    st = an.subject_settling_time(prof, cfg, cohort_ref_profile=cohort_reference)
    st["subject_id"] = sid
    st["duration_s"] = loaded_files[sid].parse_report["duration_s"]
    settle_rows.append(st)
# dsp.display(settle_rows)

## PROBLEMA - Tempo de estabilização

In [7]:
for prof in profiles_rows:
    n = len(prof)
    k = max(2, min(int(cfg.n_terminal_blocks), n // 2))
    tail = prof.iloc[-k:]
    rms_ref = float(np.median(tail["rms_counts"])) + 1e-30
    M = an.profile_matrix(prof)
    if cfg.settling_ref_mode == "cohort" and cohort_reference is not None:
        prof_ref = np.asarray(cohort_reference, float)
        ref_used = "cohort_terminal_blocks"
    else:
        prof_ref = np.median(an.profile_matrix(tail), axis=0)
        ref_used = "self_terminal_blocks"
    dev_log2 = np.abs(np.log2((prof["rms_counts"].to_numpy(float) + 1e-30) / rms_ref))
    dev_js = np.array([an.jensen_shannon(M[i], prof_ref) for i in range(n)])
    stable = (dev_log2 <= cfg.settle_tol_log2) & (dev_js <= cfg.settle_tol_js)
    


In [8]:
prof = profiles_rows[2]
n = len(prof)
k = max(2, min(int(cfg.n_terminal_blocks), n // 2))
tail = prof.iloc[-k:]
rms_ref = float(np.median(tail["rms_counts"])) + 1e-30
M = an.profile_matrix(prof)
if cfg.settling_ref_mode == "cohort" and cohort_reference is not None:
    prof_ref = np.asarray(cohort_reference, float)
    ref_used = "cohort_terminal_blocks"
else:
    prof_ref = np.median(an.profile_matrix(tail), axis=0)
    ref_used = "self_terminal_blocks"
dev_log2 = np.abs(np.log2((prof["rms_counts"].to_numpy(float) + 1e-30) / rms_ref))
dev_js = np.array([an.jensen_shannon(M[i], prof_ref) for i in range(n)])
stable = (dev_log2 <= cfg.settle_tol_log2) & (dev_js <= cfg.settle_tol_js)
print(stable)

[ True  True  True  True False  True  True  True  True  True  True  True
 False  True  True  True  True  True  True  True  True False  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True False  True  True False  True  True  True  True  True  True
  True  True  True  True  True False  True  True False  True  True  True
 False  True  True]


### Entendendo o que faz o subject_settling_time

In [27]:
prof = profiles_rows[0]
n=len(prof)
k=max(2,min(cfg.n_terminal_blocks, n//2))
tail = prof.iloc[-k:]
rms_ref = np.median(tail['rms_counts'])+1e-30
average = np.average(tail['rms_counts'])+1e-30
M = an.profile_matrix(prof)
prof_ref = np.asarray(cohort_reference, float);
ref_used = "cohort_terminal_blocks"
dev_log2 = np.abs(np.log2((prof["rms_counts"].to_numpy(float) + 1e-30) / rms_ref))
dev_js = np.array([an.jensen_shannon(M[i], prof_ref) for i in range(n)])
stable = (dev_log2 <= cfg.settle_tol_log2) & (dev_js <= cfg.settle_tol_js)
# print(dev_log2)
print(dev_log2 <= cfg.settle_tol_log2)
print("================================")
print(dev_js <= cfg.settle_tol_js)
print("================================")
print(stable)

[False False False False False False False False False False False False
 False False  True  True False  True  True  True  True False  True False
 False  True False  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True False False  True  True]
[ True False  True  True False False  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True]
[False False False False False False False False False False False False
 False False  True  True False  True  True  True  True False  True False
 False  True False  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True 

In [28]:
idx = len(stable)
for i in range(len(stable) - 1, -1, -1):
    if stable[i]:
        idx = i
    else:
        break

print('==========IDX ESTÁVEL==========')
print(str(idx)+' / '+str(len(stable)))

==========IDX ESTÁVEL==========
58 / 60


In [11]:
t = float(prof["t_start_s"].iloc[idx]) if idx < len(prof) else float("nan")
tail_rho = float(sc.stats.spearmanr(tail["t_start_s"], tail["rms_counts"]).statistic) \
        if k >= 3 else float("nan")
spans = bool(np.isfinite(tail_rho) and abs(tail_rho) >= cfg.tail_trend_warn)
print(t)
print(tail_rho)
print(spans)

1000.0
-0.7999999999999999
True


# Decisão da coorte

In [12]:
qc_df = an.cohort_qc_decision(pd.DataFrame(quality_control), cfg)
dsp.display(qc_df)

,subject_id,source_name,task,n_samples,duration_s,finite,flat_channels,n_wraparound_fixed,saturation_fraction,zero_diff_fraction,line_noise_ratio_raw,line_noise_ratio_post,notch_applied,ocular_index,muscle_ratio,nonstationarity_cv,min_channel_corr,rms_counts,log_rms_counts,z__line_noise_ratio_post,z__ocular_index,z__muscle_ratio,z__log_rms_counts,qc_reasons,qc_pass
0,02010001,02010001_still.txt,still,301740,1206.960,True,0,0,0.000000,0.000030,128.297032,0.000288,True,0.255427,0.063209,3.362686,0.973712,9.006954e+02,2.954578,5.551957,-1.577690,0.435679,0.832654,cohort_outlier_line_noise_ratio_post(z=5.6),False
1,02010002,02010002_still.txt,still,298260,1193.040,True,0,0,0.000000,0.000009,835.977175,0.000286,True,0.422666,0.075356,0.783745,0.740264,4.422825e+02,2.645700,5.501437,-0.494503,0.831637,-0.655372,cohort_outlier_line_noise_ratio_post(z=5.5),False
2,02010003,02010003_still.txt,still,316467,1265.868,True,0,0,0.000000,0.000047,4.680124,0.000003,True,0.619680,0.027674,10.228956,0.973998,1.708829e+03,3.232699,-0.753186,0.781529,-0.722606,2.172505,,True
3,02010005,02010005_still.txt,still,294409,1177.636,True,0,0,0.000000,0.000058,44.835857,0.000039,True,0.686378,0.044868,1.639146,0.852388,7.924017e+02,2.898945,0.035704,1.213524,-0.162151,0.564643,,True
4,02010006,02010006_still.txt,still,301468,1205.872,True,0,0,0.000000,0.000013,738.611386,0.000099,True,0.502529,0.086076,4.704563,0.440884,6.506734e+02,2.813363,1.373498,0.022758,1.181077,0.152349,,True
5,02010007,02010007_still.txt,still,306072,1224.288,True,0,0,0.000000,0.000025,205.336204,0.000127,True,0.352931,0.073662,2.149367,0.856803,5.959132e+02,2.775183,1.990705,-0.946172,0.776408,-0.031584,,True
6,02010008,02010008_still.txt,still,300859,1203.436,True,0,0,0.000000,0.000016,575.940071,0.000071,True,0.493178,0.061383,0.429606,0.924952,5.296662e+02,2.724002,0.751887,-0.037809,0.376168,-0.278148,,True
7,02010009,02010009_still.txt,still,303294,1213.176,True,0,0,0.000000,0.000009,1509.292190,0.000117,True,0.297743,0.060293,1.112543,0.916513,4.405748e+02,2.644020,1.768804,-1.303615,0.340646,-0.663465,,True
8,02010010,02010010_still.txt,still,300454,1201.816,True,0,0,0.000000,0.000010,861.451543,0.000063,True,0.444079,0.035854,0.866831,0.853394,5.229954e+02,2.718498,0.558796,-0.355816,-0.455980,-0.304665,,True
9,02010011,02010011_still.txt,still,303343,1213.372,True,0,137320,0.150896,0.000016,55.728372,0.002654,True,0.278362,0.237233,5.152161,-0.000003,1.624384e+07,7.210689,57.913371,-1.429143,6.108220,21.336530,saturation=0.1509; cohort_outlier_line_noise_r...,False


# QC registro de distribuição

In [13]:
qc_dist = an.qc_distribution_report(qc_df, cfg)
# dsp.display(qc_df)
dsp.display(qc_dist)

,metric,threshold_param,threshold_value,direction,n_valid,min,p05,q25,median,q75,p95,max,n_excluded_by_threshold,frac_excluded_by_threshold
0,line_noise_ratio_post,max_line_noise_ratio,0.2500,greater,55,0.000001,0.000005,0.000013,0.000037,0.000122,0.000914,0.003054,0,0.000000
1,ocular_index,max_ocular_index,0.8000,greater,55,0.167101,0.247024,0.407759,0.499015,0.642447,0.777342,0.870642,3,0.054545
2,muscle_ratio,max_muscle_ratio,0.6000,greater,55,0.001261,0.011136,0.028194,0.049843,0.070229,0.190965,0.253164,0,0.000000
3,zero_diff_fraction,max_zero_diff_fraction,0.2000,greater,55,0.000003,0.000006,0.000017,0.000047,0.000107,0.000739,0.002622,0,0.000000
4,saturation_fraction,max_saturation_fraction,0.0001,greater,55,0.000000,0.000000,0.000000,0.000000,0.000000,0.227635,0.711260,7,0.127273
5,nonstationarity_cv,max_nonstationarity_cv,inf,greater,55,0.224018,0.427590,0.739372,1.296368,4.481744,8.589636,10.228956,0,0.000000
6,min_channel_corr,min_channel_corr,-0.5000,less,55,-0.079528,-0.017954,0.795164,0.878790,0.930225,0.984900,0.999147,0,0.000000
7,z__line_noise_ratio_post,cohort_mad_z_max,4.0000,abs_greater,55,-0.794387,-0.720241,-0.528146,0.000000,1.879755,19.395574,66.767000,11,0.200000
8,z__ocular_index,cohort_mad_z_max,4.0000,abs_greater,55,-2.149768,-1.632115,-0.591054,0.000000,0.928986,1.802684,2.406977,0,0.000000
9,z__muscle_ratio,cohort_mad_z_max,4.0000,abs_greater,55,-1.583569,-1.261695,-0.705684,0.000000,0.664525,4.600063,6.627492,4,0.072727


In [14]:
settle_df = pd.DataFrame(settle_rows)
blocks_df = pd.concat(profiles_rows, ignore_index=True)
fs_df = pd.DataFrame(sampling_verification)

# ETAPA 2 - Derivação do protocolo temporal

In [15]:
durations = {s: r.parse_report["duration_s"] for s, r in loaded_files.items()}
protocol = an.derive_temporal_protocol(settle_df, durations, cfg)
windows = protocol["sensitivity_windows"]
primary_start, primary_len = windows[0]

/Users/daniel/github/eeg/analysis.py:158: RuntimeWarning: B4: inicio derivado (1160 s) excede max_skip_seconds (420 s) e foi truncado; a janela pode conter transitorio de acomodacao.
  warnings.warn("B4: inicio derivado (%.0f s) excede max_skip_seconds (%.0f s) e foi "


# Etapa 3 - Propriedades por Janela

In [16]:

feat_by_window: Dict[str, pd.DataFrame] = {}
print(windows)
# print(loaded_files)
for (s0, L) in windows:
    rows = []
    for sid, rec in loaded_files.items():
        if sid not in protocol["eligible_subjects"]:
            continue
        try:
            # print(filtered_data[sid].shape)
            rows.append(ut.extract_features_for_window(filtered_data[sid], rec, cfg, s0, L))
        except Exception as exc:
            print(f"Failed to extract features for subject {sid} in window starting at {s0} with length {L}: {exc}")
            failures.append({"file": rec.source_name, "subject_id": sid,
                                "stage": "features_w%.0f" % s0,
                                "error": "%s: %s" % (type(exc).__name__, str(exc)[:200])})
    feat_by_window["w_%.0f_%.0f" % (s0, L)] = pd.DataFrame(rows)

[(420.0, 240.0), (678.0, 240.0), (937.0, 240.0)]
Failed to extract features for subject 02010005 in window starting at 937.0 with length 240.0: Sinal insuficiente: 293409 < 294250 amostras.
Failed to extract features for subject 02030001 in window starting at 937.0 with length 240.0: Sinal insuficiente: 293649 < 294250 amostras.


In [17]:
key_primary = "w_%.0f_%.0f" % (primary_start, primary_len)
features = feat_by_window[key_primary]

# Etapa 4 - Pareamento e coorte analítica

In [18]:
merged = features.merge(metadata, on="subject_id", how="left")
unmatched = merged.loc[merged["label"].isna(), "subject_id"].tolist()
merged = merged.loc[merged["label"].notna()].copy()
merged["label"] = merged["label"].astype(int)

qc_pass = set(qc_df.loc[qc_df["qc_pass"], "subject_id"])
merged["qc_pass"] = merged["subject_id"].isin(qc_pass)
drift_ok = ((pd.to_numeric(merged.get("drift_log2_rms"), errors="coerce")
                <= cfg.max_within_window_drift_log2)
            & (pd.to_numeric(merged.get("drift_js"), errors="coerce")
                <= cfg.max_within_window_js))
merged["drift_pass"] = drift_ok.fillna(False)
analytic = merged.loc[merged["qc_pass"] & merged["drift_pass"]].reset_index(drop=True)

excl = merged.loc[~(merged["qc_pass"] & merged["drift_pass"])]
tab = pd.crosstab(merged["label"], merged["qc_pass"] & merged["drift_pass"])
fisher_p = (float(sc.stats.fisher_exact(tab.to_numpy())[1])
            if tab.shape == (2, 2) else float("nan"))

In [19]:

# B6: falhas segregadas por estagio; a v23 somava leitura e janela no mesmo total,
# de modo que um sujeito podia ser contado ate 3 vezes como "arquivo que falhou".
fail_df = pd.DataFrame(failures)
if not fail_df.empty and "stage" in fail_df.columns:
    by_stage = fail_df.groupby("stage").size().to_dict()
    subj_by_stage = {k: int(v["subject_id"].nunique())
                        for k, v in fail_df.groupby("stage")} if "subject_id" in fail_df else {}
    n_read_fail = int(fail_df.loc[fail_df["stage"].isin(
        ["discovery", "read_or_profile"])].shape[0])
else:
    by_stage, subj_by_stage, n_read_fail = {}, {}, 0
# print(len(loaded_files), len(failures), n_read_fail, by_stage, subj_by_stage)
consort = {"n_files_found": len(files),
            "n_failures_total_events": int(len(failures)),
            "n_failures_read_stage": n_read_fail,
            "failures_by_stage": by_stage,
            "n_unique_subjects_failed_by_stage": subj_by_stage,
            "n_records_read": len(loaded_files),
            "cohort_audit": ut.audit_expected_cohort(list(loaded_files), cfg),
            "n_excluded_short_duration": len(protocol["excluded_short"]),
            "excluded_short_ids": protocol["excluded_short"],
            "n_with_features_primary_window": int(len(features)),
            "n_unmatched_metadata": len(unmatched), "unmatched_ids": unmatched,
            "n_after_qc_and_drift": int(len(analytic)),
            "n_excluded_qc_or_drift": int(len(excl)),
            "excluded_qc_ids": excl["subject_id"].tolist(),
            "differential_exclusion_fisher_p": fisher_p,
            "label_distribution_analytic": analytic["label"].value_counts().to_dict()}

# Etapa 5 - gravacao dos artefatos de integracao e qualidade

In [20]:
fail_df.to_csv(output_directories['integration'] / "failures.csv", index=False)
merged.to_csv(output_directories['integration'] / "features_primary_window.csv", index=False)
for k, v in feat_by_window.items():
    v.to_csv(output_directories['sensitivity'] / ("features_%s.csv" % k), index=False)
qc_df.to_csv(output_directories['quality'] / "subject_qc.csv", index=False)
qc_dist.to_csv(output_directories['quality']/ "qc_distribution.csv", index=False)           # B5
fs_df.to_csv(output_directories['quality']/ "sampling_rate_verification.csv", index=False)  # B2
settle_df.to_csv(output_directories['quality'] / "temporal_settling.csv", index=False)
blocks_df.to_csv(output_directories['quality'] / "temporal_blocks.csv", index=False)

In [21]:
fs_summary = {"fs_assumed_hz": float(cfg.sampling_frequency),
                "fs_is_assumption": bool(cfg.fs_is_assumption),
                "n_subjects_supporting_fs": int(fs_df.get("fs_supported",
                                                        pd.Series(dtype=bool)).sum()),
                "n_subjects_evaluated": int(len(fs_df)),
                "status_counts": (fs_df["fs_verification"].value_counts().to_dict()
                                if "fs_verification" in fs_df else {}),
                "note": ("A ausencia de pico de rede NAO refuta fs; deixa a suposicao sem "
                        "corroboracao. Se fs estiver errada, todas as bandas e todo o eixo "
                        "temporal escalam por um fator constante.")}

summary: Dict[str, Any] = {
    "version": __version__, "mode": cfg.policy_mode,
    "selection_criterion": selection_criterion,
    # "config_fingerprint": fingerprint(),
    "sampling_rate_verification": fs_summary,
    "qc_thresholds_frozen": asdict(cfg.qc_freeze),
    "qc_threshold_impact": qc_dist.to_dict("records"),
    "temporal_protocol": protocol, "consort": consort,
    "open_limitations": list(OPEN_LIMITATIONS)}
if not protocol.get("protocol_valid", True):
    summary["protocol_warning"] = (
        "B4: o inicio derivado foi truncado por max_skip_seconds; a janela pode "
        "conter transitorio de acomodacao. Resultados temporais nao sao validos "
        "sob a regra declarada.")


# Etapa 6 - Análise

In [22]:
ok_cohort = (len(analytic) >= cfg.min_subjects_for_ml
        and analytic["label"].nunique() == 2
        and analytic["label"].value_counts().min() >= cfg.min_per_class_for_ml)

In [23]:
if cfg.mode == "integration" or not ok_cohort:
    summary["blocked_analyses"] = list(BLOCKED_ANALYSES)
    summary["block_reason"] = ("integration mode" if cfg.mode == "integration"
                                else "coorte insuficiente")
else:
    acfg = cfg
    conf = ut.confirmatory_group_tests(analytic, acfg)
    conf.to_csv(output_directories['analysis'] / "confirmatory_tests.csv", index=False)
    adj = ut.confound_adjusted_models(analytic, acfg)
    adj.to_csv(output_directories['analysis'] / "confound_adjusted_models.csv", index=False)
    batch = ut.batch_confound_report(analytic, acfg)
    cols = ut.primary_columns(analytic)
    X = analytic[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    y = analytic["label"].to_numpy(int)
    res = ut.nested_cv_scores(X, y, acfg)
    met = ut.metrics_from_probs(y, res["prob_mean"], acfg,
                                oof_per_repeat=res["oof_prob_per_repeat"])  # B10
    covs = [c for c in acfg.covariates if c in analytic.columns]
    base = {"majority_class": {"balanced_accuracy": 0.5, "roc_auc": 0.5}}
    if covs:
        Xd = analytic[covs].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        if np.isfinite(Xd).any():
            rd = ut.nested_cv_scores(Xd, y, acfg)
            base["demographic"] = ut.metrics_from_probs(y, rd["prob_mean"], acfg,
                                                        rd["oof_prob_per_repeat"])
            rc = ut.nested_cv_scores(np.hstack([X, Xd]), y, acfg)
            base["eeg_plus_demographic"] = ut.metrics_from_probs(
                y, rc["prob_mean"], acfg, rc["oof_prob_per_repeat"])
        else:
            base["demographic"] = {"error": "covariaveis sem valores finitos (ver B9)"}
    # B1: observado e nulo sob o MESMO estimador
    obs_matched = ut.matched_observed_auc(X, y, acfg)
    perm = ut.permutation_auc_test(X, y, obs_matched, acfg)

    # B8: TODAS as janelas com o mesmo numero de repeticoes da primaria
    reps_sens = int(acfg.sensitivity_outer_repeats or acfg.outer_repeats)
    sens = {}
    for k, dfw in feat_by_window.items():
        m = dfw.merge(metadata, on="subject_id", how="inner")
        m = m.loc[m["subject_id"].isin(analytic["subject_id"])]
        if len(m) < cfg.min_subjects_for_ml or m["label"].nunique() < 2:
            continue
        ck = ut.confirmatory_group_tests(m, acfg)
        Xk = m[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        yk = m["label"].to_numpy(int)
        rk = ut.nested_cv_scores(Xk, yk, acfg, repeats=reps_sens)
        sens[k] = {"n": int(len(m)), "cv_repeats": reps_sens,
                    "roc_auc": float(roc_auc_score(yk, rk["prob_mean"])),
                    "effects": ck.set_index("feature")["hedges_g"].to_dict(),
                    "n_significant_fdr": int(ck["significant_fdr"].sum())}
        ck.to_csv(output_directories['sensitivity'] / ("confirmatory_%s.csv" % k), index=False)

    # B7: concordancia com IC e p binomial, sem veredito qualitativo
    gref = conf.set_index("feature")["hedges_g"]
    concord = {}
    for k, v in sens.items():
        if k == key_primary:
            continue
        c = ut.sign_agreement_with_ci(gref, pd.Series(v["effects"]),
                                    n_boot=acfg.sign_agreement_n_boot,
                                    seed=acfg.random_seed)
        c["roc_auc"] = v["roc_auc"]
        concord[k] = c

    summary["analysis"] = {
        "primary_window": {"start_s": primary_start, "seconds": primary_len},
        "cv_repeats_all_windows": reps_sens,
        "classification_eeg_primary": met, "baselines": base,
        "permutation_test": perm,
        "n_significant_fdr": int(conf["significant_fdr"].sum()),
        "confound_adjusted_available": bool(len(adj)),
        "batch_confound": batch, "window_sensitivity": sens,
        "window_concordance": concord,
        "interpretation_guard": (
            "Metricas sao de validacao interna em um unico conjunto, com poder para "
            "detectar apenas efeitos grandes (g >= 0,78 com 26 vs 28). "
            + ("Lote e diagnostico sao colineares (V = 1,00): NAO interprete como "
                "efeito clinico." if batch.get("batch_confounded") else ""))}
    json.dump(summary["analysis"], open(output_directories['analysis'] / "analysis_summary.json", "w"),
                indent=2, ensure_ascii=False, default=ut.json_default)


In [24]:
print(len(files), len(loaded_files), len(features), len(merged), len(analytic))
dsp.display(merged[["qc_pass","drift_pass"]].value_counts(dropna=False))
display(merged[["drift_log2_rms","drift_js"]].describe())   # NaN? faixa?
display(merged[["drift_log2_rms","drift_js"]].isna().sum())
display(qc_dist)                                             # exclusões por limiar
display(fail_df.groupby("stage").size() if not fail_df.empty else "sem falhas")
print(len(unmatched), unmatched[:10])                      # ID não casou com metadados
display(pd.crosstab(merged["label"], merged["qc_pass"] & merged["drift_pass"]))

55 55 54 54 35


qc_pass  drift_pass
True     True          35
False    True           9
         False          8
True     False          2
Name: count, dtype: int64

,drift_log2_rms,drift_js
count,54.000000,54.000000
mean,1.415566,0.072505
std,2.612237,0.069934
min,0.241721,0.007984
25%,0.594325,0.031453
50%,0.867086,0.046090
75%,1.246282,0.082711
max,19.541451,0.333250


drift_log2_rms    0
drift_js          0
dtype: int64

,metric,threshold_param,threshold_value,direction,n_valid,min,p05,q25,median,q75,p95,max,n_excluded_by_threshold,frac_excluded_by_threshold
0,line_noise_ratio_post,max_line_noise_ratio,0.2500,greater,55,0.000001,0.000005,0.000013,0.000037,0.000122,0.000914,0.003054,0,0.000000
1,ocular_index,max_ocular_index,0.8000,greater,55,0.167101,0.247024,0.407759,0.499015,0.642447,0.777342,0.870642,3,0.054545
2,muscle_ratio,max_muscle_ratio,0.6000,greater,55,0.001261,0.011136,0.028194,0.049843,0.070229,0.190965,0.253164,0,0.000000
3,zero_diff_fraction,max_zero_diff_fraction,0.2000,greater,55,0.000003,0.000006,0.000017,0.000047,0.000107,0.000739,0.002622,0,0.000000
4,saturation_fraction,max_saturation_fraction,0.0001,greater,55,0.000000,0.000000,0.000000,0.000000,0.000000,0.227635,0.711260,7,0.127273
5,nonstationarity_cv,max_nonstationarity_cv,inf,greater,55,0.224018,0.427590,0.739372,1.296368,4.481744,8.589636,10.228956,0,0.000000
6,min_channel_corr,min_channel_corr,-0.5000,less,55,-0.079528,-0.017954,0.795164,0.878790,0.930225,0.984900,0.999147,0,0.000000
7,z__line_noise_ratio_post,cohort_mad_z_max,4.0000,abs_greater,55,-0.794387,-0.720241,-0.528146,0.000000,1.879755,19.395574,66.767000,11,0.200000
8,z__ocular_index,cohort_mad_z_max,4.0000,abs_greater,55,-2.149768,-1.632115,-0.591054,0.000000,0.928986,1.802684,2.406977,0,0.000000
9,z__muscle_ratio,cohort_mad_z_max,4.0000,abs_greater,55,-1.583569,-1.261695,-0.705684,0.000000,0.664525,4.600063,6.627492,4,0.072727


stage
features_w937    2
dtype: int64

0 []


col_0,False,True
label,,
0,7,21
1,12,14


In [25]:
an.audit_nonstationarity_redundancy(merged, qc_df)


===== AUDITORIA DE REDUNDÂNCIA (QC vs DRIFT) =====
Sujeitos com drift_pass=True mas qc_pass=False: 9
  -> Falham ESTRITAMENTE por nonstationarity_cv: 0
  -> Falham por nonstationarity + outros motivos: 0
  -> Falham apenas por outros motivos de QC:     9
N amostral se nonstationarity for delegado ao drift: 35



{'n_target_qc_fail_drift_pass': 9,
 'n_failed_ONLY_by_nonstationarity': 0,
 'ids_failed_ONLY_by_nonstationarity': [],
 'n_failed_by_nonstat_plus_others': 0,
 'n_failed_by_other_qc_reasons': 9,
 'potential_analytic_n': 37}